# Credit Underwriting with SparkRules

This notebook walks through building a complete credit underwriting pipeline:
1. Define underwriting rules in DRL
2. Evaluate applicants against the rules
3. Generate adverse-action notices for declined applicants (ECOA/FCRA compliance)
4. Profile applicant data for quality checks

```bash
pip install sparkrules
```

In [ ]:
from sparkrules.executor.local_executor import LocalRuleExecutor
from sparkrules.executor import build_adverse_action_notice
from sparkrules.dq.profile import profile_rows
from sparkrules.compiler.rulepack import RulePack
import json

## Step 1: Define underwriting rules

Rules use Drools-style DRL syntax. Key concepts:
- **salience**: priority (higher = evaluated first)
- **activation-group**: mutual exclusion (only one rule fires per group)
- **reason_codes**: regulatory codes attached to adverse decisions
- **when/then**: condition/action pattern

In [ ]:
UNDERWRITING_RULES = """
rule "auto-decline-fico-below-580"
  salience 100
  activation-group "decision"
  reason_codes ["CR001", "FICO_VERY_LOW"]
  when
    $app : Application( $app.fico < 580 )
  then
    result.decision = "DECLINE";
    result.reason = "FICO score below minimum threshold";
end

rule "decline-high-dti"
  salience 90
  reason_codes ["DTI001"]
  when
    $app : Application( $app.dti > 0.50 )
  then
    result.dti_flag = "EXCESSIVE";
    result.dti_reason = "DTI exceeds 50% threshold";
end

rule "refer-marginal-fico"
  salience 50
  activation-group "decision"
  reason_codes ["CR002"]
  when
    $app : Application( $app.fico < 660 )
  then
    result.decision = "REFER";
    result.reason = "Marginal FICO - manual review";
end

rule "refer-low-income"
  salience 45
  reason_codes ["IN001"]
  when
    $app : Application( $app.income < 35000 )
  then
    result.income_flag = "LOW";
end

rule "approve-prime"
  salience 10
  activation-group "decision"
  when
    $app : Application( $app.fico >= 740 )
  then
    result.decision = "APPROVE";
    result.tier = "PRIME";
    result.rate_adj = 0;
end

rule "approve-standard"
  salience 5
  activation-group "decision"
  when
    $app : Application( $app.fico >= 660 )
  then
    result.decision = "APPROVE";
    result.tier = "STANDARD";
    result.rate_adj = 50;
end
"""

# Inspect the rule pack classification
pack = RulePack.from_drl(UNDERWRITING_RULES)
print(f"Rules: {len(pack.rules)}")
print(f"SQL_PUSHDOWN: {len(pack.sql_pushdown)}")
print(f"ALPHA_SHARED: {len(pack.alpha_shared)}")
print(f"PYTHON_FALLBACK: {len(pack.python_fallback)}")
for r in pack.rules:
    print(f"  {r.name} (sal={r.salience}): {r.strategy.name}")

## Step 2: Build the executor and evaluate applicants

In [ ]:
executor = LocalRuleExecutor.from_rulepack(pack)

applicants = [
    {"id": "APP-001", "app": {"fico": 550, "dti": 0.55, "income": 28000}},
    {"id": "APP-002", "app": {"fico": 640, "dti": 0.35, "income": 52000}},
    {"id": "APP-003", "app": {"fico": 780, "dti": 0.25, "income": 120000}},
    {"id": "APP-004", "app": {"fico": 700, "dti": 0.40, "income": 45000}},
    {"id": "APP-005", "app": {"fico": 620, "dti": 0.48, "income": 30000}},
]

for applicant in applicants:
    result = executor.score(applicant)
    decision = result.merged_actions.get("decision", "NO_DECISION")
    fired = [f.rule_name for f in result.fires if f.fired]
    print(f"{applicant['id']} FICO={applicant['app']['fico']:>3d} DTI={applicant['app']['dti']:.0%} -> {decision:>10s}  fired: {fired}")

## Step 3: Generate adverse-action notice

ECOA (Equal Credit Opportunity Act) and FCRA (Fair Credit Reporting Act) require
lenders to provide up to 4 principal reasons when declining a credit application.

In [ ]:
# Evaluate the declined applicant
declined = executor.score(applicants[0])

# Build adverse-action notice from fired rules
notice = build_adverse_action_notice(
    [f for f in declined.fires if f.fired],
    decision="DECLINE",
    fact_id=applicants[0]["id"],
    max_reasons=4,
)

print("Adverse Action Notice")
print("=" * 40)
print(json.dumps(notice.to_dict(), indent=2))

## Step 4: Profile applicant data quality

In [ ]:
facts = [a["app"] for a in applicants]
profile = profile_rows(facts)

print(f"Rows: {profile.total_rows}, Fields: {profile.total_fields}\n")
for f in profile.fields:
    line = f"{f.field_name:>10s}: {f.completeness:.0%} complete, {f.uniqueness:.0%} unique"
    if f.numeric_stats:
        ns = f.numeric_stats
        line += f"  mean={ns.mean:,.0f}  range=[{ns.min_val}, {ns.max_val}]"
    print(line)